### Install depencies

In [2]:
!pip install unsloth bitsandbytes transformers accelerate
!pip install gradio
!pip install PyMuPDF  # for PDF reading

#### Import packages

In [3]:
import unsloth
unsloth.__version__

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


'2026.3.8'

In [4]:
import accelerate
accelerate.__version__

'1.13.0'

In [5]:
import transformers
transformers.__version__

'5.3.0'

In [6]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

### Import model from Hugging Face

- Login into HuggingFace

In [8]:
from huggingface_hub import notebook_login
notebook_login()

- Import model

In [9]:
model_id = "aman012/mistral-7b-instruct-v0.3-bnb-4bit-200"
MAX_SEQ_LENGTH = 2048
dtype = None
load_in_4bit = True

In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.3.8: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Unsloth 2026.3.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [11]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096, padding_idx=770)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

- Dummy Example

In [13]:
text = """Large Language Models(LLMs) are essential tools in natural language processing (NLP) and have been used in a variety of applications, such as text completion, translation, and question answering.

The output of large language models can be affected by various hyperparameters including temperature, top p, token length, max tokens and stop tokens.
Temperature

Temperature is a hyperparameter that controls the randomness of language model output.

A high temperature produces more unpredictable and creative results, while a low temperature produces more deterministic and conservative output. In other words, a higher temperature setting causes the model to be more “confident” in its output. A lower temperature setting yields more conservative and predictable output.

For example, if you adjust the temperature to 0.5, the model will generate text that is more predictable and less creative than if you set the temperature to 1.0.
Top p

Top p, also known as nucleus sampling, is another hyperparameter that controls the randomness of language model output.

It sets a threshold probability and selects the top tokens whose cumulative probability exceeds the threshold. The model then randomly samples from this set of tokens to generate output. This method can produce more diverse and interesting output than traditional methods that randomly sample the entire vocabulary.

For example, if you set top p to 0.9, the model will only consider the most likely words that make up 90% of the probability mass.
Token length

This is the number of words or characters in a sequence or text that is fed to the LLM.

It varies depending on the language and the tokenization method used for the particular LLM.

The length of the input text affects the output of the LLM.

A very short input may not have enough context to generate a meaningful completion.

Conversely, a rather long input may make the model inefficiently process or it may cause the model to generate an irrelevant output.
Max tokens

This is the maximum number of tokens that the LLM generates.

Within this, is the token limit; the maximum number of tokens that can be used in the prompt and the completion of the model. Determined by the architecture of the model LLM, it refers to the maximum tokens that can be processed at once.

The computational cost and the memory requirements are directly proportional to the max tokens. Set a longer max token, and you will have greater context and coherent output text. Set a shorter max token, and you will use less memory and have a faster response but your output is prone to errors and inconsistencies.

During the training and fine-tuning of the LLM, the max token is set.

Contrary to fine-tuning token length during the generation of output, the coherence and length of the output is carefully set at inception, based on the specific task & requirements, without affecting other parameters that will likely need adjusting.
Stop tokens

In simple terms, it is the length of the output or response of an LLM.

So it signifies the end of a sequence in terms of either a paragraph or a sentence.

Similar to max tokens, the inference budget is reduced when the stop tokens are set low.

For example, when the stop tokens are set at 2, the generated text or output will be limited to a paragraph. If the stop tokens is set at 1, the generated text will be limited to a sentence.
Other relevant hyperparameters

There are many other hyperparameters that can affect large language model performance, such as learning rate, number of layers, hidden size, and dropout rate. These hyperparameters can impact the model’s training time, accuracy, and generalizability."""

### Define functions

- Extract text from pdf function

In [14]:
import fitz  # first install fPyMuPDF
#Extract text from pdf
print(fitz.__version__)
def extract_text_from_pdf(file):
    doc = fitz.open(file)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


1.27.2.2


- Split text into chunks function

We have trained out model with max_token 32768, number_of_token > 2048 will exceeds model's maximum sequence length. So we have to divide whole pdf into small chunks

In [15]:
def split_text_to_chunks(text: str, max_token: int = 1600):
    tokens = tokenizer.tokenize(text) #split text into tokens
    chunks = []
    for i in range(0, len(tokens), max_token):
        chunk = tokenizer.convert_tokens_to_string(tokens[i:i+max_token])
        chunks.append(chunk)
    return chunks

- Summarize text function

In [16]:
import torch
torch.__version__

'2.10.0+cu128'

In [17]:
def summarize_text(text: str, summary_len: int, tempt: float, top_p: float):
    if not text or not text.strip():
        return "❗ Please enter some text before summarizing."
    try:
        prompt = f"Summarize the following text into a concise paragraph of about {summary_len} words:\n\n{text}\n\nSummary:"
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = summary_len * 2,
                temperature = tempt,
                do_sample = False,
                top_p = top_p
            )

            Summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
            return Summary.replace(prompt, "").strip()
    except Exception as e:
        return f"An error occurred during summarization: {str(e)}"

In [18]:
summarize_text(text, 30, 0.9, 0.9)

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

"An error occurred during summarization: output with shape [1, 32, 1, 128] doesn't match the broadcast shape [1, 32, 834, 128]"

- Summarize pdf function

In [ ]:
import tqdm
tqdm.__version__

'4.67.1'

In [ ]:
#Summarization
from typing import Any
from tqdm import tqdm #use to see the progress bar of every chunk
def summarize_pdf(file: Any, summary_len: int, tempt: float, top_p: float):
    pdf_text = extract_text_from_pdf(file)
    if not pdf_text or not pdf_text.strip():
        return "Please upload any pdf first !"
    try:
        prompt = f"Summarize the following text into a concise paragraph of about {summary_len} words:\n\n{pdf_text}\n\nSummary:"

        # split into chunks
        chunks = split_text_to_chunks(pdf_text)

        #summarize each chunks
        summary_chunks = []
        for chunk in tqdm(chunks,  desc="Summarizing chunks"):
            summary = summarize_text(chunk, summary_len, tempt, top_p)
            summary_chunks.append(summary)

        # combine all summaries
        Summary = ' '.join(summary_chunks)
        final_summary = summarize_text(Summary, summary_len, tempt, top_p)
        return final_summary
    except Exception as e:
        return f"An error occurred during summarization: {str(e)}"

In [ ]:
summarize_pdf('/content/attention-2-3-1.pdf', 100, 0.8, 0.5)

Summarizing chunks: 100%|██████████| 1/1 [00:16<00:00, 16.60s/it]


'The Transformer is a model architecture that relies entirely on an attention mechanism to draw global dependencies between input and output. It allows for significant more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. The Transformer follows an encoder-decoder structure, with the encoder composed of a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. The decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. The Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence-aligned RNNs or convolution.'

#### Build a web interface using Gradio

In [ ]:
import gradio
gradio.__version__

'5.29.0'

In [ ]:
import gradio as gr
with gr.Blocks() as demo:
    gr.Markdown("## 🧠 Smart PDF/Text Summarizer with LLM\nUpload a file or type text. Customize your summary!")
    with gr.Tab("Type text"):
        text_input = gr.Textbox(lines=10, placeholder="Input your text here", label="Text Input")
        text_output = gr.Textbox(max_lines=100, placeholder="Show your output here", show_copy_button=True, label="Output")
        summary_len1 = gr.Slider(20, 200, value=30, step=1, label="length")
        temperature1 = gr.Slider(0.1, 1.5, value=0.8, step=0.1, label="Temperature")
        top_p1 = gr.Slider(0.1, 1.0, value=0.8, step=0.1, label="Top-p (nucleus sampling)")
        text_btn = gr.Button("Summarize Text")
        text_btn.click(fn=summarize_text, inputs=[text_input, summary_len1, temperature1, top_p1], outputs=text_output)
    with gr.Tab("upload pdf"):
        pdf_input = gr.File(label="Upload PDF")
        pdf_output = gr.Textbox(max_lines=100, placeholder="Show your output here", show_copy_button=True, label="Output")
        summary_len2 = gr.Slider(20, 200, value=30, step=1, label="length")
        temperature2 = gr.Slider(0.4, 1.3, value=0.7, step=0.1, label="Temperature")
        top_p2 = gr.Slider(0.6, 1.0, value=0.5, step=0.1, label="Top-p (nucleus sampling)")
        pdf_output = gr.Textbox(max_lines=100, placeholder="Show your output here", show_copy_button=True, label="Output")
        pdf_btn = gr.Button("Summarize PDF")
        pdf_btn.click(fn=summarize_pdf, inputs= [pdf_input, summary_len2, temperature2, top_p2], outputs=pdf_output)

    demo.launch(share = True, debug = True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9dea28ff67e4ab56fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Summarizing chunks: 100%|██████████| 1/1 [00:12<00:00, 12.36s/it]


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9dea28ff67e4ab56fd.gradio.live
